# CityGML → Sionna RT via Blender

**What this notebook is:** a step-by-step recipe for taking a CityGML building model
already imported into Blender, preparing it for a Sionna RT radio-propagation
simulation, and exporting it in two formats: a Mitsuba XML scene (for Sionna itself)
and an FBX file (a lightweight reference file so a teammate can open it in Blender
and pick camera / antenna locations, without needing any of the CityGML/Sionna
tooling installed).

**Important, how to actually run this:** this is *not* a live, run-the-whole-notebook
Jupyter file. `bpy` (Blender's Python API) only exists inside Blender's own Python
interpreter, not in a normal Jupyter kernel. Instead:

- **Section 1 (Manual)**: copy each code cell below, one at a time, into Blender's
  built-in *Scripting* workspace (or the *Python Console*), and run it there, in order.
  This works for anyone with Blender open. No AI assistant, no special setup required.
- **Section 2 (AI-assisted)**: if you have an AI coding assistant connected to Blender
  via MCP (e.g. via the `blender-mcp` bridge), you can instead just describe
  the goal and let it run these same steps for you interactively. Useful because it can
  adapt to a different scene, check its own work, and fix small data problems as it goes
  (see the troubleshooting note in 1.5 for an example of exactly that happening).
- **Section 3 (Verify)**: a real Jupyter/Python section (this one *does* run with a
  normal kernel, unlike Sections 1 and 2) that loads your exported file back into
  Sionna RT, so you can confirm the export actually works before handing it off.

Read Section 1 first even if you plan to use Section 2. It explains *why* each step
exists, which the AI-assisted version leans on.

## 0. Before you start

You need, inside Blender:

1. **The [CityGML importer/exporter add-on](https://github.com/orttak/blender-citygml-importer-exporter)**
   installed and enabled, with a CityGML file already imported into your
   scene (buildings visible as mesh objects). This notebook picks up *after*
   that import. It doesn't do the import itself, since that's a one-click
   step in the add-on's own panel (`File > Import`). If your team uses a
   *different* CityGML add-on, check what custom property names it actually
   writes for surface type before trusting §1.3 below. This notebook was
   built against the add-on linked above, which writes `surface_type`,
   `SurfaceTyp`, `Typ`, or `FeatureType` depending on context.
2. **The [Mitsuba-Blender export add-on](https://github.com/mitsuba-renderer/mitsuba-blender)**
   installed and enabled (needed for step 1.6, the Sionna-facing export).

**Blender version:** this pipeline has been run successfully on Blender
4.1. The Mitsuba-Blender add-on's own docs call out 3.6 and 4.2 (both LTS
releases) as their most-tested versions rather than 4.1. If you hit an
add-on install or compatibility issue, trying 4.2 LTS is a reasonable first
move before debugging further. See the root README's Setup section for the
full version matrix (Blender, both add-ons, `sionna-rt`).

You do **not** need Sionna itself installed to run Section 1 or 2. Sionna
only comes into play in Section 3, where the exported `.xml` file gets
loaded back in to confirm it works.

---
# Section 1: Manual workflow (Blender Scripting tab / Python Console)

Open Blender, switch to the **Scripting** workspace tab at the top (or open a
**Python Console** area), and run each of the following cells in order by pasting
the code in and pressing the run/execute button (or Enter for the console).

## 1.1 Why material *names* matter: the `itu_` naming trick

Sionna RT ships a library of built-in radio materials based on the ITU-R P.2040-3
recommendation (concrete, brick, glass, wood, metal, various ground types, etc.).
You don't need to hand-write physics parameters for these. You just need to get a
Blender material *named exactly* the right thing, and the rest happens automatically,
for a very specific technical reason worth knowing:

- When Blender's Mitsuba exporter writes out a material, it names it in the exported
  file as `mat-<the Blender material's name>` (this is fixed exporter behavior, not
  something we configure).
- Sionna's scene loader specifically recognizes exported material names of the form
  `mat-itu_<type>` and automatically swaps in its built-in ITU material with the
  correct physics for `<type>`.

So: naming a Blender material `itu_concrete` is genuinely all that's required. No XML
editing, no Python material classes. The valid `<type>` values (append `itu_` to any of
these to get the Blender material name) are:

`concrete`, `brick`, `wood`, `metal`, `glass`, `marble`, `floorboard`, `ceiling_board`,
`chipboard`, `plasterboard`, `plywood`, `very_dry_ground`, `medium_dry_ground`,
`wet_ground`.

**Note for anyone maintaining an older material-assignment script:** if you have an
older version of this pipeline that only sets `obj.material_slots[0].material`, be
aware that CityGML buildings typically have **dozens of material slots per object**
(one per imported surface polygon). Touching only slot `0` silently leaves almost
every other surface on the building unchanged. Everything below loops over *every*
slot on *every* object.

## 1.2 Create the ground plane

**This is a phase-1 stand-in, not real terrain.** The plane is a flat
rectangle sized and positioned from the imported buildings' own bounding
box, not real topography. It exists so Sionna has *something* to bounce
rays off below the buildings, not to model Hamburg's actual ground surface.

How it's built:
- **X/Y extent:** the bounding box of every imported mesh object, expanded
  by `XY_OFFSET` (15 m default) on every side, so the plane extends a bit
  past the outermost building rather than stopping exactly at its edge.
- **Z height:** the *highest* of each building's own *lowest* point. This
  guarantees no building floats with an empty gap above the plane (the
  building whose base sits highest touches the plane exactly). The
  tradeoff is that buildings whose real base is lower than this will have
  their lowest part poke through the plane. Accepted for now (Hamburg,
  phase 1); revisit once a real terrain model is in place (see below).

**Roadmap for replacing this** (noted here so the tradeoff above doesn't
get forgotten):
1. *(this cell)* Bounding-box-derived flat plane. Good enough to remove the
   "empty gap under every building" bug, still not real terrain.
2. Real topography via [BlenderGIS](https://github.com/domlysz/BlenderGIS)
   (SRTM/DEM import), geolocated using the CityGML source's own CRS/EPSG
   code so the terrain and the buildings line up in real-world coordinates.
   **First-draft implementation exists:**
   [`terrain_via_blendergis_optional.ipynb`](./terrain_via_blendergis_optional.ipynb)
   in this same folder, built from BlenderGIS's actual source code, but not
   yet run end-to-end in a live Blender session. Its own §0 spells out the
   one real unverified assumption (whether the CityGML importer preserves
   absolute coordinates or silently recentres geometry) that needs checking
   before it can be trusted.
3. A high-resolution (1 m) DTM dataset that exists for Hamburg. Open
   point, not yet scoped.
4. Road infrastructure data. Longer-term, not relevant yet.

Adjust `XY_OFFSET` below if 15 m isn't the right margin for your scene.

In [ ]:
import bpy
import bmesh
from mathutils import Vector

XY_OFFSET = 15  # meters, ground plane extends this far beyond the building
                # bounding box in every X/Y direction. See 1.2 above.

if "Ground" in bpy.data.objects:
    bpy.data.objects.remove(bpy.data.objects["Ground"], do_unlink=True)

# Bounding box of everything currently imported (the CityGML buildings).
# World-space, since an object's local bound_box ignores its transform.
building_objs = [obj for obj in bpy.data.objects if obj.type == 'MESH']
if not building_objs:
    raise RuntimeError(
        "No mesh objects found in the scene. Did the CityGML import "
        "happen before running this cell? See 0. Before you start.")

min_x = min_y = float('inf')
max_x = max_y = float('-inf')
lowest_z_per_building = []

for obj in building_objs:
    world_corners = [obj.matrix_world @ Vector(corner) for corner in obj.bound_box]
    xs = [v.x for v in world_corners]
    ys = [v.y for v in world_corners]
    zs = [v.z for v in world_corners]
    min_x = min(min_x, *xs)
    max_x = max(max_x, *xs)
    min_y = min(min_y, *ys)
    max_y = max(max_y, *ys)
    lowest_z_per_building.append(min(zs))

# X/Y: building bounding box + XY_OFFSET margin on every side.
min_x -= XY_OFFSET
max_x += XY_OFFSET
min_y -= XY_OFFSET
max_y += XY_OFFSET

# Z: the highest of each building's own lowest point. See 1.2 above for
# why (avoids an empty gap between any building and the plane, at the cost
# of lower buildings poking through it slightly).
ground_z = max(lowest_z_per_building)

bm = bmesh.new()
verts = [
    (min_x, min_y, ground_z),
    (max_x, min_y, ground_z),
    (max_x, max_y, ground_z),
    (min_x, max_y, ground_z),
]
bm_verts = [bm.verts.new(v) for v in verts]
bm.faces.new(bm_verts)

mesh = bpy.data.meshes.new("Ground")
bm.to_mesh(mesh)
bm.free()

obj = bpy.data.objects.new("Ground", mesh)
bpy.context.collection.objects.link(obj)

print(f"Ground plane created from {len(building_objs)} mesh objects:")
print(f"  X [{min_x:.1f}, {max_x:.1f}]  Y [{min_y:.1f}, {max_y:.1f}]  Z = {ground_z:.2f}")
print(f"  ({XY_OFFSET}m XY margin; Z snapped to the highest per-building minimum)")

## 1.3 Assign Sionna materials by CityGML surface type

CityGML-aware Blender importers tag every imported surface with what kind of surface
it is: a custom property (readable as `material["surface_type"]`, sometimes named
`SurfaceTyp` or `Typ` depending on the importer/locale) with a value like
`"WallSurface"`, `"RoofSurface"`, `"GroundSurface"`, or `"OuterCeilingSurface"`. This is
far more reliable than guessing material from geometry (e.g. "is this face pointing
up?") because it comes straight from the source CityGML data, not a geometric guess.

Edit the `MAPPING` dictionary below to change which ITU material each surface type
gets. The values used for the Hamburg dataset this pipeline was built against:

In [ ]:
# Edit this mapping to taste. Left = CityGML surface_type tag, right = Blender
# material name to assign (must be a valid itu_<type> name, see section 1.1).
MAPPING = {
    "WallSurface": "itu_brick",
    "RoofSurface": "itu_concrete",
    "GroundSurface": "itu_concrete",       # building footprint / underside, not the terrain
    "OuterCeilingSurface": "itu_concrete",  # undersides of overhangs, balconies, etc.
}
GROUND_OBJECT_MATERIAL = "itu_very_dry_ground"  # for the standalone terrain plane from 1.2
FALLBACK_MATERIAL = "itu_concrete"  # used if a surface has no recognizable tag at all

import bpy

materials = bpy.data.materials
target_materials = {}
for name in set(list(MAPPING.values()) + [GROUND_OBJECT_MATERIAL, FALLBACK_MATERIAL]):
    target_materials[name] = materials.get(name) or materials.new(name=name)

slot_counts = {}

for obj in bpy.data.objects:
    if obj.type != 'MESH':
        continue
    # IMPORTANT: loop every slot, not just slot 0. See the note in 1.1.
    for slot in obj.material_slots:
        if slot.material is None:
            continue
        surface_type = (slot.material.get("surface_type")
                         or slot.material.get("SurfaceTyp")
                         or slot.material.get("Typ"))
        target_name = MAPPING.get(surface_type, FALLBACK_MATERIAL)
        slot.material = target_materials[target_name]
        slot_counts[target_name] = slot_counts.get(target_name, 0) + 1

# Handle the standalone Ground plane separately. It has no CityGML surface_type
# tag since it isn't part of the imported data.
ground_obj = bpy.data.objects.get("Ground")
if ground_obj:
    while ground_obj.material_slots:
        ground_obj.material_slots.remove(ground_obj.material_slots[0])
    ground_obj.data.materials.append(target_materials[GROUND_OBJECT_MATERIAL])
    slot_counts[GROUND_OBJECT_MATERIAL] = slot_counts.get(GROUND_OBJECT_MATERIAL, 0) + 1

# Clean up: the CityGML import typically creates one unique material per surface
# polygon (tens of thousands of them). After reassigning slots above, almost all of
# those original materials are no longer referenced by anything, so remove them so
# the file (and the eventual export) stays small.
removed = 0
for mat in list(bpy.data.materials):
    if mat.users == 0:
        bpy.data.materials.remove(mat)
        removed += 1

print("Slots reassigned per material:", slot_counts)
print(f"Orphaned original materials cleaned up: {removed}")

## 1.3b Make materials visually distinguishable (optional, cosmetic only)

The two cells below are purely for *your own eyes* while working in Blender. They
don't change simulation physics at all, only how materials look in the viewport and
in renders. Useful for a quick visual sanity-check of 1.3 before you export: does the
building actually look like walls-vs-roof-vs-ground the way you'd expect?

1. Assigns each `itu_` material a distinct pastel color (Blender's *Viewport Display*
   color plus the Principled BSDF *Base Color*, so it shows up both in solid shading
   and in Material Preview / Rendered shading).
2. Sets the World background to a neutral, evenly-lit gray so materials aren't
   distorted by a dark or colored background while you're reviewing them.

Safe to skip entirely if you don't care about the visual check.

In [ ]:
# Assign a distinct pastel viewport color to every itu_ material that exists in the
# scene. Covers the full set of valid ITU material names (see 1.1); harmless if a
# given one isn't used in your scene, it's just skipped.

import bpy

PASTEL_PALETTE = {
    "itu_brick":              (0.85, 0.55, 0.55, 1.0),  # terracotta
    "itu_concrete":           (0.75, 0.78, 0.80, 1.0),  # pale slate gray
    "itu_glass":              (0.65, 0.85, 0.85, 0.4),  # translucent mint/teal
    "itu_metal":              (0.50, 0.65, 0.75, 1.0),  # muted steel blue
    "itu_wood":               (0.82, 0.73, 0.63, 1.0),  # warm beige
    "itu_marble":             (0.90, 0.87, 0.80, 1.0),  # pale cream
    "itu_floorboard":         (0.80, 0.65, 0.45, 1.0),  # warm plank tan
    "itu_ceiling_board":      (0.88, 0.88, 0.82, 1.0),  # off-white
    "itu_chipboard":          (0.70, 0.60, 0.45, 1.0),  # tan/brown
    "itu_plasterboard":       (0.92, 0.90, 0.85, 1.0),  # near-white
    "itu_plywood":            (0.78, 0.68, 0.50, 1.0),  # light wood
    "itu_very_dry_ground":    (0.76, 0.65, 0.45, 1.0),  # sandy tan
    "itu_medium_dry_ground":  (0.55, 0.45, 0.30, 1.0),  # medium brown earth
    "itu_wet_ground":         (0.30, 0.28, 0.22, 1.0),  # dark damp earth
}

applied = 0
for mat_name, color_rgba in PASTEL_PALETTE.items():
    mat = bpy.data.materials.get(mat_name)
    if not mat:
        continue  # this material isn't used in the current scene, nothing to color
    mat.diffuse_color = color_rgba
    if mat.use_nodes and mat.node_tree:
        principled = mat.node_tree.nodes.get("Principled BSDF")
        if principled:
            principled.inputs['Base Color'].default_value = color_rgba
    applied += 1

for area in bpy.context.screen.areas:
    if area.type == 'VIEW_3D':
        area.tag_redraw()

print(f"Applied pastel colors to {applied} materials present in the scene")

In [ ]:
# Set a clean, neutral World background so materials aren't visually distorted
# during review (e.g. by Blender's default dark gray world).

import bpy

if not bpy.context.scene.world:
    bpy.context.scene.world = bpy.data.worlds.new("Review_World")

world = bpy.context.scene.world
world.use_nodes = True
nodes = world.node_tree.nodes

bg_node = nodes.get("Background") or next((n for n in nodes if n.type == 'BACKGROUND'), None)
if bg_node:
    bg_node.inputs['Color'].default_value = (0.85, 0.85, 0.85, 1.0)  # clean studio gray
    bg_node.inputs['Strength'].default_value = 1.0
    print("World background set to neutral gray, strength 1.0")
else:
    print("Warning: could not find a Background shader node to configure")

## 1.4 Exclude fine architectural details from export

LOD3 CityGML data often includes a lot of small detail objects: balconies,
chimneys, stairs, dormers, antennas, tagged with `feature_type == "BuildingInstallation"`
(as opposed to the main building shells, tagged `"Building"`). These add a lot of
geometric complexity that's usually overkill for a radio propagation simulation, so
it's common to exclude them from export.

We do this with the `hide_render` flag (the camera icon next to each object in the
Outliner). This is **non-destructive**: nothing is deleted, the objects stay exactly
as they are in the `.blend` file, they're just skipped by the exporters in 1.6/1.7.
Set `hide_render = False` again any time you want them back for a higher-detail run
(the cell right after shows how).

In [ ]:
import bpy

count = 0
for obj in bpy.data.objects:
    if obj.type == 'MESH' and obj.get("feature_type") == "BuildingInstallation":
        obj.hide_render = True
        count += 1

print(f"hide_render set on {count} BuildingInstallation objects (reversible, nothing deleted)")

*Optional: to bring them back later for a more detailed run:*

In [ ]:
# Undo 1.4: re-include BuildingInstallation objects in future exports.
import bpy

count = 0
for obj in bpy.data.objects:
    if obj.type == 'MESH' and obj.get("feature_type") == "BuildingInstallation":
        obj.hide_render = False
        count += 1

print(f"hide_render cleared on {count} BuildingInstallation objects")

## 1.5 Repair geometry before exporting (run this even if you don't think you need it)

Real-world CityGML data sometimes contains tiny data-quality defects, most commonly
two identical overlapping faces sharing the same vertices, often facing opposite
directions. This causes Blender's computed normal at the shared vertices to cancel out
to exactly zero, and Mitsuba's exporter **refuses to export** a mesh with a zero-length
normal, failing with an error like:

```
RuntimeError: [BlenderMesh] Error while loading Blender mesh "...": invalid normals!
```

This cell finds and removes exact duplicate faces scene-wide and recalculates normals.
It's cheap to run and safe even if your data has no such issues (it will simply report
0 objects fixed). Running it as a standard step avoids a failed export followed by a
confusing debugging detour. (This is exactly what happened during development of this
pipeline: the very first export attempt failed with the error above, traced to 2
buildings out of ~40 having 3 duplicate faces between them, a small, contained issue,
but one that blocks the export entirely until fixed.)

In [ ]:
import bpy
import bmesh

objects_fixed = 0
total_dupes_removed = 0

for obj in bpy.data.objects:
    if obj.type != 'MESH' or obj.hide_render:
        continue  # skip objects already excluded in 1.4, no need to repair what we won't export

    mesh = obj.data
    bm = bmesh.new()
    bm.from_mesh(mesh)
    bm.faces.ensure_lookup_table()

    seen = {}
    dupe_faces = []
    for f in bm.faces:
        sig = frozenset(v.index for v in f.verts)
        if sig in seen:
            dupe_faces.append(f)
        else:
            seen[sig] = f.index

    if dupe_faces:
        bmesh.ops.delete(bm, geom=dupe_faces, context='FACES')
        bmesh.ops.recalc_face_normals(bm, faces=bm.faces)
        bm.to_mesh(mesh)
        mesh.update()
        objects_fixed += 1
        total_dupes_removed += len(dupe_faces)

    bm.free()

print(f"Objects with duplicate faces removed: {objects_fixed}")
print(f"Total duplicate faces removed: {total_dupes_removed}")

## 1.6 Export to Mitsuba XML (the file Sionna actually loads)

**Critical detail, easy to get wrong:** the Mitsuba exporter's *default* axis
convention converts Blender's Z-up coordinates into Mitsuba's usual Y-up convention
(used by most renderers). Sionna RT, however, expects **Z-up preserved**: its example
scenes and coordinate conventions (transmitter/receiver positions, etc.) all assume Z
is height. If you export with the exporter's defaults, the whole scene will be silently
rotated 90 degrees relative to what Sionna expects, with no error, just wrong results.

The `axis_forward='Y', axis_up='Z'` arguments below override this and keep Blender's
native coordinates unchanged in the export. Always use these settings for a Sionna-bound
export.

Edit `EXPORT_PATH` below to wherever you want the `.xml` file (and its companion
`meshes/` folder of `.ply` files) written.

In [ ]:
import bpy

EXPORT_PATH = r"C:\path\to\your\export\scene_name.xml"  # <-- edit this

bpy.ops.export_scene.mitsuba(
    filepath=EXPORT_PATH,
    use_selection=False,   # False = export everything not hidden via hide_render (1.4)
    axis_forward='Y',      # Sionna-correct: keep Blender's Z-up, do not convert
    axis_up='Z',
    export_ids=True,       # lets Sionna reference objects by name later
    split_files=False,
)
print("Mitsuba export done:", EXPORT_PATH)

## 1.7 Export to FBX (for picking camera / antenna locations)

This is a plain, lightweight geometry file that any teammate can open directly in
Blender (or most other 3D tools) without installing the CityGML or Mitsuba add-ons.
Useful for someone whose job is just "look at the scene and tell us where the cameras
/ antennas should go", not run the simulation itself.

We apply the **same** `axis_forward='Y', axis_up='Z'` override here as in 1.6. This
matters because Blender's FBX exporter has its own separate default (aimed at the
generic FBX-ecosystem Y-up convention, e.g. for Unity/Unreal), which would otherwise
rotate the FBX relative to the Mitsuba export. With matching axis settings on both
exports, any position someone picks in the FBX file (e.g. by placing an Empty object
and reading its X/Y/Z coordinates) is directly usable as a Sionna transmitter/receiver
position, with no manual coordinate conversion. (Section 3 shows exactly this: reusing
a picked position directly as a Sionna `Camera`.)

In [ ]:
import bpy

FBX_EXPORT_PATH = r"C:\path\to\your\export\scene_name.fbx"  # <-- edit this

bpy.ops.export_scene.fbx(
    filepath=FBX_EXPORT_PATH,
    use_visible=True,      # only export objects not hidden via hide_render (1.4)
    axis_forward='Y',      # match the Mitsuba export's axis convention, see note above
    axis_up='Z',
)
print("FBX export done:", FBX_EXPORT_PATH)

---
# Section 2: AI-assisted workflow (MCP-connected assistants)

**What MCP is, in one sentence:** Model Context Protocol (MCP) is a standard way for
an AI coding assistant to connect to and control other running applications. In this
case, a small add-on inside Blender opens a socket that the assistant talks to, letting
it run Python code inside your live Blender session and see the results (scene
contents, screenshots, etc.), instead of you copy-pasting code by hand.

This is entirely optional. Section 1 above is the complete, self-contained recipe.
Section 2 is for teammates who have an AI coding assistant connected to Blender via
an MCP bridge (commonly the `blender-mcp` add-on) and would rather describe the goal
and let the assistant carry out the steps, adapting them to their specific scene.

### What the assistant can do that a manual copy-paste can't
The main advantage isn't speed, it's **verification and adaptation**. For example,
during the development of this exact pipeline, the first Mitsuba export attempt failed
with an "invalid normals" error (see 1.5). An AI assistant with MCP access can:
- see the real error message and the specific mesh it names,
- inspect that mesh's geometry directly to diagnose *why* (duplicate faces, in this
  case) rather than guessing,
- write and run a targeted fix,
- retry, and confirm the fix worked (e.g. by re-checking object/material counts or
  taking a viewport screenshot),

all without you needing to know what a "degenerate face" is. A person running Section
1 manually would just see the export fail and need to debug it themselves (or ask for
help).

### How to actually use this section
If your assistant is connected to Blender via MCP, just describe the goal in your own
words and point it at this notebook for the specifics, for example:

> "Using the steps in `gml_to_sionna_via_blender.ipynb` section 1 as a reference,
> prepare the CityGML scene currently open in Blender for a Sionna simulation: derive
> a ground plane from the imported buildings' own bounding box with a 15m margin
> (see 1.2, this is a flat stand-in, not real terrain), assign Sionna ITU materials
> by CityGML surface type (walls = brick, everything else = concrete, ground =
> very_dry_ground), exclude BuildingInstallation detail objects, and export both a
> Mitsuba XML (for Sionna) and an FBX (for picking camera locations) to
> `<your path here>`. Check for and fix any geometry errors that come up during
> export."

The assistant should still follow the same logic documented in Section 1 (correct
material naming, looping every slot, the axis-convention override, etc.). This
notebook is what it should be checked against if its output looks wrong.

### One privacy note if you set up `blender-mcp` yourself
The common community `blender-mcp` add-on sends anonymous usage telemetry to a
third-party service by default. If you'd rather it didn't (recommended for anything
involving non-public project data), set the environment variable
`BLENDER_MCP_DISABLE_TELEMETRY=true` wherever the MCP server process is configured,
before connecting it to your assistant.

---
# Section 3: Verify the export actually works (real Jupyter/Python, run here)

Unlike Sections 1 and 2, **this section runs in a normal Python/Jupyter kernel**, not
inside Blender. Its only job is to close the loop: load the file you just exported in
1.6 back into Sionna RT, and confirm it's valid before you hand it off or hard-code it
into a bigger simulation script. Catching a broken export here (empty file, wrong
path, scene that fails to parse) takes seconds; catching it three steps into a real
simulation run does not.

In [ ]:
# Tell Mitsuba which compute backend to use before importing sionna.rt.
# The official Sionna RT tutorial (https://nvlabs.github.io/sionna/rt/tutorials/Introduction.html)
# is explicit that if you set a variant manually, it must end in
# `_ad_mono_polarized`, not `_ad_rgb`. 'cuda_ad_mono_polarized' uses the
# GPU (recommended if you have an NVIDIA card); if you don't, or hit issues,
# 'llvm_ad_mono_polarized' runs on CPU instead. Whatever you pick here must
# match what step 2 uses, or results won't be comparable.

import mitsuba as mi
mi.set_variant('cuda_ad_mono_polarized')

import sionna.rt
from sionna.rt import load_scene, Camera
print(f"Mitsuba variant in use: {mi.variant()}")

The cell below points at the file exported in 1.6 and does a few defensive checks
before attempting to load it. An empty or half-written XML file (e.g. from an export
that was interrupted, or a typo'd path) fails with a fairly cryptic
`ParseError: no element found` otherwise, which doesn't obviously point back at
"the export didn't actually finish".

In [ ]:
import os

SCENE_XML_PATH = r"C:\path\to\your\export\scene_name.xml"  # <-- same path as EXPORT_PATH in 1.6

if not os.path.isfile(SCENE_XML_PATH):
    raise FileNotFoundError(f"Scene XML not found: {SCENE_XML_PATH}")
if os.path.getsize(SCENE_XML_PATH) == 0:
    raise ValueError(f"Scene XML is empty, the export likely didn't finish: {SCENE_XML_PATH}")

with open(SCENE_XML_PATH, "r", encoding="utf-8", errors="ignore") as f:
    head = f.read(256).lstrip()
    if not head.startswith("<"):
        print(f"Warning: {SCENE_XML_PATH} does not look like XML, check the export step.")

scene = load_scene(SCENE_XML_PATH)
print(f"Loaded OK. Object count: {len(scene.objects)}")
print("Object names:", list(scene.objects.keys())[:10], "..." if len(scene.objects) > 10 else "")

## Optional: render a quick look from a chosen viewpoint

If your hiwi picked a camera position in the FBX from 1.7 (e.g. by placing an Empty
object and reading off its X/Y/Z in Blender), that position is directly usable here.
The axis conventions were matched between the two exports specifically so this works
without any conversion.

`preview()` opens an interactive 3D widget and only works in an actual Jupyter
notebook (not e.g. a plain terminal script). The `no_preview` flag below lets the
same cell fall back to a static render in either case.

In [ ]:
no_preview = False  # set True to always use the static render path instead of the
                    # interactive widget (e.g. running headless / outside Jupyter)

# Replace with the position picked in the FBX (1.7). Same coordinate system, no conversion needed.
my_cam = Camera(position=[-30, 50, 30], look_at=[-15, -30, 2])

if not no_preview:
    scene.preview()
else:
    scene.render(camera=my_cam, resolution=[650, 500], num_samples=512)

If that renders (or previews) without errors and the buildings look right, the
export is good to build a real simulation on top of. Transmitter/receiver placement,
path solving, radio maps, etc. are their own, separate topic (see Sionna's own
[tutorials](https://nvlabs.github.io/sionna/rt/tutorials.html) for that next step).

---
## Appendix: troubleshooting quick reference

| Symptom | Cause | Fix |
|---|---|---|
| Export fails with `invalid normals!` | Duplicate/overlapping faces in the source geometry | Run the repair cell in 1.5 |
| Scene looks rotated / sideways after export | Forgot the `axis_forward='Y', axis_up='Z'` override | Re-export with those settings (1.6 / 1.7) |
| Sionna can't find `itu-radio-material` plugin | Very old Sionna version (pre-1.0) using a different material system | This notebook targets `sionna-rt` 2.x (validated against 2.0.1, the version the [official tutorial](https://nvlabs.github.io/sionna/rt/tutorials/Introduction.html) currently uses); the `itu_<type>` naming approach in 1.1 is for that version |
| Only some surfaces on a building got a material | An older script only touched `material_slots[0]` | Use the loop in 1.3, which covers every slot |
| MCP tools (Section 2) not showing up in your assistant | A newly-added MCP server usually needs a full restart of the assistant/IDE, not just a "reconnect" | Fully restart, then retry |
| `ParseError: no element found` when loading in Section 3 | The XML file is empty or wasn't fully written (interrupted or failed export) | Re-run 1.6, check it prints "Mitsuba export done" with no exception, then retry Section 3 |
| Manually set Mitsuba variant doesn't match | Any variant used with Sionna RT must end in `_ad_mono_polarized`, not `_ad_rgb` (see Section 3) | Use `cuda_ad_mono_polarized` (GPU) or `llvm_ad_mono_polarized` (CPU), and use the same one in step 2 |